# Lab 07 — Support Vector Machines and Kernel Methods
## Notebook 01 — In-class Demonstration

This notebook uses **synthetic 2D data** so that SVM decision boundaries can be inspected directly. We use scikit-learn throughout; the SVM optimization algorithm is not implemented manually.

Main ideas:
- preprocessing and scaling;
- linear SVM and support vectors;
- Linear, Polynomial, and RBF kernels;
- the effect of `C`, `gamma`, and polynomial `degree`.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42

## 2. Helper functions for repeated experiments

These functions only automate **evaluation and plotting**. The SVM model itself is still scikit-learn's `SVC`.

In [ ]:
def svm_pipeline(kernel, C=1.0, degree=3, gamma="scale"):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel=kernel, C=C, degree=degree, gamma=gamma))
    ])

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    support_count = int(model.named_steps["svm"].n_support_.sum())
    return {
        "Model": name,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Support Vectors": support_count
    }

def plot_boundary(model, X, y, title):
    plt.figure(figsize=(7, 5))
    DecisionBoundaryDisplay.from_estimator(
        model,
        X,
        response_method="predict",
        alpha=0.25
    )
    plt.scatter(X[:, 0], X[:, 1], c=y, s=28)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)
    plt.show()

# Part A — Approximately Linear Data

We first generate a two-class dataset that is approximately linearly separable. To make preprocessing visible, the second feature is deliberately placed on a much larger numerical scale.

In [ ]:
X_linear, y_linear = make_blobs(
    n_samples=300,
    centers=[(-2, -2), (2, 2)],
    cluster_std=1.4,
    random_state=RANDOM_STATE
)

# Deliberately create very different feature scales.
X_linear[:, 1] = X_linear[:, 1] * 80

linear_df = pd.DataFrame(X_linear, columns=["Feature_1", "Feature_2"])
linear_df["Target"] = y_linear

display(linear_df.head())
print("Shape:", linear_df.shape)
print()
print("Missing values:")
print(linear_df.isna().sum())
print()
print("Duplicates:", linear_df.duplicated().sum())
print()
print("Class counts:")
print(linear_df["Target"].value_counts().sort_index())
display(linear_df[["Feature_1", "Feature_2"]].describe())

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_linear[:, 0], X_linear[:, 1], c=y_linear, s=28)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Synthetic Linear Dataset — Original Scale")
plt.show()

## 3. Train/test split and scaling

The split is created **before** the scaler learns any parameters. Using a `Pipeline` ensures that `StandardScaler` is fitted only on training data and then applied consistently to test data.

In [ ]:
X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_linear,
    y_linear,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_linear
)

linear_svm = svm_pipeline(kernel="linear")
linear_svm.fit(X_train_lin, y_train_lin)

print("Training accuracy:", accuracy_score(y_train_lin, linear_svm.predict(X_train_lin)))
print("Test accuracy:", accuracy_score(y_test_lin, linear_svm.predict(X_test_lin)))
print("Support vectors by class:", linear_svm.named_steps["svm"].n_support_)

## 4. Visualize the linear boundary and support vectors

In [ ]:
plt.figure(figsize=(7, 5))
DecisionBoundaryDisplay.from_estimator(
    linear_svm,
    X_linear,
    response_method="decision_function",
    plot_method="contour",
    levels=[-1, 0, 1]
)
plt.scatter(X_linear[:, 0], X_linear[:, 1], c=y_linear, s=28)

scaler = linear_svm.named_steps["scaler"]
svc = linear_svm.named_steps["svm"]
support_original_scale = scaler.inverse_transform(svc.support_vectors_)
plt.scatter(
    support_original_scale[:, 0],
    support_original_scale[:, 1],
    s=120,
    marker="x"
)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Linear SVM — Boundary, Margins, and Support Vectors")
plt.show()

The points highlighted with `x` are the training observations used as **support vectors**. They are the points that play the key role in defining the fitted margin/boundary.

## 5. Compare Linear, Polynomial, and RBF kernels on the same linear dataset

In [ ]:
kernel_models_linear = {
    "Linear": svm_pipeline("linear"),
    "Polynomial": svm_pipeline("poly", degree=3),
    "RBF": svm_pipeline("rbf")
}

linear_results = []
for name, model in kernel_models_linear.items():
    linear_results.append(
        evaluate_model(name, model, X_train_lin, X_test_lin, y_train_lin, y_test_lin)
    )

linear_results = pd.DataFrame(linear_results).sort_values("Test Accuracy", ascending=False)
display(linear_results)

In [ ]:
for name, model in kernel_models_linear.items():
    model.fit(X_train_lin, y_train_lin)
    plot_boundary(model, X_linear, y_linear, f"Linear Dataset — {name} Kernel")

# Part B — Curved Nonlinear Data: make_moons

A straight separating line is now a much stronger restriction. We compare the same three kernels.

In [ ]:
X_moons, y_moons = make_moons(
    n_samples=350,
    noise=0.20,
    random_state=RANDOM_STATE
)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moons, y_moons,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_moons
)

plt.figure(figsize=(7, 5))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, s=28)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("make_moons Dataset")
plt.show()

In [ ]:
moon_models = {
    "Linear": svm_pipeline("linear"),
    "Polynomial": svm_pipeline("poly", degree=3),
    "RBF": svm_pipeline("rbf")
}

moon_results = []
for name, model in moon_models.items():
    moon_results.append(
        evaluate_model(name, model, X_train_m, X_test_m, y_train_m, y_test_m)
    )
    plot_boundary(model, X_moons, y_moons, f"make_moons — {name} Kernel")

moon_results = pd.DataFrame(moon_results).sort_values("Test Accuracy", ascending=False)
display(moon_results)

# Part C — Radial Nonlinear Data: make_circles

The two classes form an inner/outer structure. This is difficult to represent using a single linear hyperplane in the original two-dimensional space.

In [ ]:
X_circles, y_circles = make_circles(
    n_samples=350,
    noise=0.08,
    factor=0.40,
    random_state=RANDOM_STATE
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_circles, y_circles,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_circles
)

plt.figure(figsize=(7, 5))
plt.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, s=28)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("make_circles Dataset")
plt.show()

In [ ]:
circle_models = {
    "Linear": svm_pipeline("linear"),
    "Polynomial": svm_pipeline("poly", degree=3),
    "RBF": svm_pipeline("rbf")
}

circle_results = []
for name, model in circle_models.items():
    circle_results.append(
        evaluate_model(name, model, X_train_c, X_test_c, y_train_c, y_test_c)
    )
    plot_boundary(model, X_circles, y_circles, f"make_circles — {name} Kernel")

circle_results = pd.DataFrame(circle_results).sort_values("Test Accuracy", ascending=False)
display(circle_results)

# Part D — Effect of C

`C` controls how strongly classification errors/margin violations are penalized.

- smaller `C`: stronger regularization and more tolerance for violations;
- larger `C`: stronger pressure to classify the training observations correctly.

In [ ]:
C_values = [0.01, 1, 100]
C_results = []

for C in C_values:
    model = svm_pipeline("rbf", C=C, gamma="scale")
    result = evaluate_model(
        f"RBF, C={C}", model,
        X_train_m, X_test_m, y_train_m, y_test_m
    )
    C_results.append(result)
    plot_boundary(model, X_moons, y_moons, f"RBF on make_moons — C={C}")

display(pd.DataFrame(C_results))

# Part E — Effect of gamma for the RBF kernel

`gamma` controls how local the influence of training observations is in the RBF kernel.

In [ ]:
gamma_values = [0.01, 1, 100]
gamma_results = []

for gamma in gamma_values:
    model = svm_pipeline("rbf", C=1, gamma=gamma)
    result = evaluate_model(
        f"RBF, gamma={gamma}", model,
        X_train_m, X_test_m, y_train_m, y_test_m
    )
    gamma_results.append(result)
    plot_boundary(model, X_moons, y_moons, f"RBF on make_moons — gamma={gamma}")

display(pd.DataFrame(gamma_results))

# Part F — Effect of Polynomial Degree

Increasing polynomial degree increases the expressive capacity of the polynomial kernel, but greater complexity does not guarantee better test performance.

In [ ]:
degrees = [2, 3, 5]
degree_results = []

for degree in degrees:
    model = svm_pipeline("poly", C=1, degree=degree)
    result = evaluate_model(
        f"Polynomial, degree={degree}", model,
        X_train_m, X_test_m, y_train_m, y_test_m
    )
    degree_results.append(result)
    plot_boundary(model, X_moons, y_moons, f"Polynomial on make_moons — degree={degree}")

display(pd.DataFrame(degree_results))

# 6. Final kernel comparison across synthetic datasets

The aim is not to memorize a fixed ranking of kernels. Instead, connect **data geometry** with model behaviour and then confirm the choice experimentally.

In [ ]:
summary = pd.concat([
    linear_results.assign(Dataset="Approximately linear"),
    moon_results.assign(Dataset="Moons"),
    circle_results.assign(Dataset="Circles")
], ignore_index=True)

summary = summary[["Dataset", "Model", "Train Accuracy", "Test Accuracy", "Support Vectors"]]
display(summary.sort_values(["Dataset", "Test Accuracy"], ascending=[True, False]))

## Main takeaways

- SVMs are sensitive to feature scale, so scaling should be part of the preprocessing pipeline.
- A Linear kernel is suitable when a linear separating hyperplane is sufficient.
- Polynomial and RBF kernels allow nonlinear decision boundaries.
- Support vectors are the key training observations that define the fitted SVM boundary/margin.
- `C`, `gamma`, and `degree` change model flexibility and should be selected from evidence, not by assuming that larger values are better.
- Feature correlation alone does not determine the best kernel.